# Optimized RSS Headline Sentiment Dashboard

This version is faster because it removes repeated heavyweight generation for live RSS monitoring and uses cached RSS retrieval plus deterministic headline scoring for instant updates.

## What changed

- Removed repeated LLM generation from the live dashboard path.
- Added `@st.cache_data(ttl=90)` for RSS fetches.
- Kept sentiment logic lightweight and deterministic for fast refreshes.
- Added KPI cards, bar chart, live table, and CSV export.
- Added optional 2-minute auto refresh.

In [ ]:
!pip -q install streamlit feedparser pandas

In [ ]:
import re
import time
from email.utils import parsedate_to_datetime
from urllib.parse import urlparse

import feedparser
import pandas as pd
import streamlit as st

st.set_page_config(page_title="Live RSS Sentiment Dashboard", page_icon="📰", layout="wide")

RSS_FEEDS = {
    "Google News Business (IE)": "https://news.google.com/rss/headlines/section/topic/BUSINESS?hl=en-IE&gl=IE&ceid=IE:en",
    "Google News World (IE)": "https://news.google.com/rss/headlines/section/topic/WORLD?hl=en-IE&gl=IE&ceid=IE:en",
    "Google News Technology (IE)": "https://news.google.com/rss/headlines/section/topic/TECHNOLOGY?hl=en-IE&gl=IE&ceid=IE:en",
    "Google News Top Stories (IE)": "https://news.google.com/rss?hl=en-IE&gl=IE&ceid=IE:en",
    "BBC Top Stories": "http://feeds.bbci.co.uk/news/rss.xml",
    "BBC Business": "http://feeds.bbci.co.uk/news/business/rss.xml",
}

POSITIVE_PATTERNS = [
    ("discount", 2), ("tenant", 1), ("raises rate on savings", 4), ("raises savings rate", 4),
    ("higher savings rate", 4), ("savings account", 2), ("investment", 3), ("invests", 3),
    ("expansion", 3), ("expand", 2), ("job growth", 3), ("new jobs", 3), ("growth", 2),
    ("raises forecast", 3), ("beats estimates", 3), ("higher revenue", 3), ("support scheme", 2),
    ("affordable", 2),
]
NEGATIVE_PATTERNS = [
    ("prices rose", 3), ("home prices rose", 4), ("house prices rose", 4), ("rent rises", 4),
    ("warning", 3), ("warns", 3), ("loss", 3), ("losses", 3), ("fall", 2), ("decline", 2),
    ("cuts", 2), ("layoffs", 4), ("job cuts", 4), ("misses estimates", 4), ("fine", 3),
    ("lawsuit", 3), ("probe", 3), ("downgrade", 3), ("crisis", 4), ("jet fuel left", 5),
    ("fuel left", 4), ("disruption", 4), ("supply shock", 4), ("war", 4), ("risk", 2),
]
NEUTRAL_PATTERNS = [
    ("on the market", 2), ("for sale", 2), ("live updates", 2), ("latest on", 2), ("talks", 1),
    ("statement", 1), ("meeting", 1), ("report", 1), ("appoints", 1), ("names", 1),
]

SOURCE_CLEAN = [
    r"\s*-\s*RTE\.ie\s*$", r"\s*-\s*The Irish Times\s*$", r"\s*-\s*The Irish Independent\s*$",
    r"\s*-\s*BBC News\s*$",
]

@st.cache_data(ttl=90, show_spinner=False)
def fetch_rss(feed_name: str, max_stories: int) -> pd.DataFrame:
    feed = feedparser.parse(RSS_FEEDS[feed_name])
    rows = []
    for entry in feed.entries[:max_stories]:
        title = str(getattr(entry, "title", "") or "").strip()
        if not title:
            continue
        link = str(getattr(entry, "link", "") or "").strip()
        published = str(getattr(entry, "published", "") or "").strip()
        source_name = feed_name
        source_obj = getattr(entry, "source", None)
        if source_obj:
            try:
                source_name = source_obj.get("title", feed_name) or feed_name
            except Exception:
                pass
        rows.append({"source": source_name, "headline": title, "published": published, "url": link})
    return pd.DataFrame(rows)

def normalize_headline(text: str) -> str:
    out = str(text or "").strip()
    for pat in SOURCE_CLEAN:
        out = re.sub(pat, "", out, flags=re.I)
    return out.strip(" .:-\n\t\"'")

def classify_headline(headline: str):
    h = normalize_headline(headline).lower()
    pos = neg = neu = 0
    for phrase, weight in POSITIVE_PATTERNS:
        if phrase in h:
            pos += weight
    for phrase, weight in NEGATIVE_PATTERNS:
        if phrase in h:
            neg += weight
    for phrase, weight in NEUTRAL_PATTERNS:
        if phrase in h:
            neu += weight
    if "discount" in h and "tenant" in h:
        pos += 4
    if ("home prices rose" in h or "house prices rose" in h) and "%" in h:
        neg += 3
    if "savings account" in h and "raises rate" in h:
        pos += 4
    if "jet fuel left" in h or ("six weeks" in h and "fuel" in h):
        neg += 5
    if neg >= pos + 2 and neg >= neu:
        return "Negative", "worsens economic outlook"
    if pos >= neg + 2 and pos >= neu:
        return "Positive", "supports economic outlook"
    return "Neutral", "direction remains unclear"

def format_time(value: str) -> str:
    try:
        return parsedate_to_datetime(value).strftime("%Y-%m-%d %H:%M")
    except Exception:
        return value

def analyze(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    labels, reasons = zip(*[classify_headline(h) for h in df["headline"].tolist()])
    out = df.copy()
    out["predicted"] = list(labels)
    out["reasoning"] = list(reasons)
    out["published"] = out["published"].map(format_time)
    out["domain"] = out["url"].map(lambda u: urlparse(u).netloc.replace("www.", "") if u else "")
    return out

def sentiment_emoji(label: str) -> str:
    return {"Positive":"🟢 Positive","Negative":"🔴 Negative","Neutral":"⚪ Neutral"}.get(label, "⚪ Neutral")

st.title("📰 Live RSS headline sentiment")
st.caption("Fast version: cached RSS fetch + rule-based sentiment for near-instant refreshes")

with st.sidebar:
    st.header("Controls")
    feed_name = st.selectbox("Feed", list(RSS_FEEDS.keys()), index=0)
    max_stories = st.slider("Stories", 10, 100, 30, 5)
    auto_refresh = st.checkbox("Auto refresh every 2 min", value=False)
    refresh = st.button("Refresh now", use_container_width=True)

if auto_refresh:
    st.markdown('<meta http-equiv="refresh" content="120">', unsafe_allow_html=True)

start = time.time()
if refresh:
    fetch_rss.clear()

df = analyze(fetch_rss(feed_name, max_stories))
elapsed = time.time() - start

if df.empty:
    st.warning("No stories returned from this feed.")
    st.stop()

counts = df["predicted"].value_counts().reindex(["Positive","Negative","Neutral"], fill_value=0)
col1, col2, col3, col4 = st.columns(4)
col1.metric("Stories", int(len(df)))
col2.metric("Positive", int(counts["Positive"]))
col3.metric("Negative", int(counts["Negative"]))
col4.metric("Refresh time", f"{elapsed:.2f}s")

chart_df = pd.DataFrame({"sentiment": counts.index, "count": counts.values})
st.bar_chart(chart_df.set_index("sentiment"))

show = df[["source","headline","predicted","reasoning","published","domain","url"]].copy()
show["predicted"] = show["predicted"].map(sentiment_emoji)
st.dataframe(show, use_container_width=True, hide_index=True)

csv = df.to_csv(index=False).encode("utf-8")
st.download_button("Download current results CSV", csv, file_name="rss_sentiment_snapshot.csv", mime="text/csv")

## Run locally in Colab or notebook terminal

```bash
streamlit run output/rss_sentiment_dashboard.py --server.port 8501 --server.address 0.0.0.0
```

If you still want the LLM model, keep it as an offline validation path for sampled headlines only, not for every live refresh.